# KVO_08 — Satellite prediction of Musanga canopy fraction

**Five sites:** MOS_1_2, Yoko, Yangambi, Baego, Luki.

**Phase 1: data-preparation/validation.** The first runnable cells audit existing NB04 products without changing them. The NB04 production manifest documents the four-site final frozen model, uint16 0–10000 probability and 65535 NoData. Luki is an external site. The four current NB04 maps may be annotation-ROI restricted; their *pre-vector-ROI-mask* backups have broader coverage but must not be assumed full mosaic until audited.

**CRS rule:** keep UAV source pixels in their native projected CRS (Luki EPSG:32733; other sites verified individually), and later construct per-site target grids aligned to a chosen **actual Sentinel-2 10 m projection + transform**. Satellite embeddings and S2 are *not assumed* co-aligned; verify exact grid transforms first. No reprojection is performed by the audit.

**Run Section 00 and new Section 04 first (CPU), then review earlier audits only if needed.** They only read metadata and small samples of probability maps; they never run U-Net, change any raster, or download satellites. All results go to `KVO_work/KVO_08_satellite_fraction/validation/`.

## 00 — Paths, site inventory and processing contract

**CPU / read-only.** Define native UAV sources, the current four-site/Luki outputs and optional full-coverage candidates. No source rasters are modified.

In [ ]:
# BLOCK 00 — canonical site paths; CPU-only, read-only inspection
from pathlib import Path
import json, math, shutil
import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window
from IPython.display import display
from google.colab import drive

MY = Path("/content/drive/MyDrive")
if not MY.exists(): drive.mount("/content/drive")
ROOT = MY / "Colab Notebooks" / "Musanga"
GEO = ROOT / "data" / "Geo_Ref"
KVO04 = ROOT / "KVO_work" / "KVO_04_final_multisite"
MAPS = KVO04 / "probability_maps"
OUT = ROOT / "KVO_work" / "KVO_08_satellite_fraction"
VALIDATION = OUT / "validation"
VALIDATION.mkdir(parents=True, exist_ok=True)

RGB = {
    "MOS_1_2": GEO / "MOS_1_2_kvo.tif",
    "Yoko": GEO / "Yoko_kvo_georef.tif",
    "Yangambi": GEO / "Yangambi_georef.tif",
    "Baego": GEO / "Baego_georef.tif",
    "Luki": GEO / "Luki.tif",
}
CURRENT = {site: MAPS / f"{site}_musanga_probability_uint16.tif" for site in RGB}
BACKUP = {
    site: MAPS / f"{site}_musanga_probability_uint16.pre_vector_roi_mask.tif"
    for site in RGB if site != "Luki"
}
FULL_LUKI = MAPS / "Luki_musanga_probability_uint16_FULL.tif"

NODATA = 65535
SCALE = 10000
SITES = list(RGB)
print("Validation outputs:", VALIDATION)
print("Sites:", ", ".join(SITES))
print("Luki FULL map present:", FULL_LUKI.exists())
print("No inference, overwrites, reprojecting, or downloads in Blocks 00–02.")

## 01 — Raster metadata, CRS and probability schema

**CPU / read-only.** Verify raster dimensions, pixel origins, CRS, probability encoding and NoData. Current UAV mosaics are EPSG:32635 for MOS, Yoko, Yangambi and Baego; Luki is EPSG:32733. These are *not* reprojected in this step.

In [ ]:
# BLOCK 01 — Native-CRS metadata and probability schema checks (READ-ONLY)
# Candidate types are recorded explicitly. 'pre_ROI_backup' is NOT assumed full coverage.
import warnings
def meta(path):
    with rasterio.open(path) as ds:
        return {
            "width": ds.width, "height": ds.height, "crs": str(ds.crs),
            "transform": tuple(ds.transform), "bounds": tuple(ds.bounds),
            "px_x_m": abs(ds.transform.a), "px_y_m": abs(ds.transform.e),
            "dtype": ds.dtypes[0], "bands": ds.count, "nodata": ds.nodata,
            "tags": ds.tags(),
            "bbox_ha": ds.width*ds.height*abs(
                ds.transform.a*ds.transform.e-ds.transform.b*ds.transform.d
            )/10000,
        }

rows = []
candidates = []
for site in SITES:
    rgb_path = RGB[site]
    base = {"site":site,"rgb_path":str(rgb_path),
            "rgb_exists":rgb_path.is_file(),
            "current_exists":CURRENT[site].is_file()}
    if not rgb_path.is_file():
        rows.append({**base,"status":"MISSING_RGB"})
        continue
    rgb_meta = meta(rgb_path)
    assert rgb_meta["crs"]!="None",f"{site}: RGB CRS is undefined"
    base.update(rgb_crs=rgb_meta["crs"],
                gsd_x_m=rgb_meta["px_x_m"],gsd_y_m=rgb_meta["px_y_m"],
                rgb_bands=rgb_meta["bands"],
                width=rgb_meta["width"],height=rgb_meta["height"],
                rgb_bbox_ha=rgb_meta["bbox_ha"])
    candidate_paths = [("current",CURRENT[site])]
    if site in BACKUP: candidate_paths.append(("pre_ROI_backup",BACKUP[site]))
    if site=="Luki": candidate_paths.append(("full_mosaic",FULL_LUKI))
    for kind,p in candidate_paths:
        item={**base,"candidate_kind":kind,"probability_path":str(p),
              "probability_exists":p.is_file()}
        if not p.is_file():
            item["status"]="MISSING_PROBABILITY"
            rows.append(item)
            continue
        pm=meta(p)
        same_grid=(pm["width"]==rgb_meta["width"] and
                   pm["height"]==rgb_meta["height"] and
                   pm["crs"]==rgb_meta["crs"] and
                   np.allclose(pm["transform"],rgb_meta["transform"],rtol=0,atol=1e-6))
        schema_ok=(pm["bands"]==1 and pm["dtype"]=="uint16" and
                   pm["nodata"]==NODATA)
        item.update(prob_crs=pm["crs"],prob_dtype=pm["dtype"],
                    prob_nodata=pm["nodata"],same_uav_grid=same_grid,
                    schema_ok=schema_ok,coverage_tag=pm["tags"].get("COVERAGE",""),
                    model_checkpoint=pm["tags"].get("MODEL_CHECKPOINT",""),
                    probability_file_GB=round(p.stat().st_size/1e9,3),
                    status="SCHEMA_OK" if same_grid and schema_ok else "SCHEMA_ERROR")
        rows.append(item)
        candidates.append((site,kind,p,item["status"]))
audit=pd.DataFrame(rows)
audit.to_csv(VALIDATION/"01_reference_metadata.csv",index=False)
display(audit[["site","candidate_kind","probability_exists","rgb_crs",
               "rgb_bbox_ha","same_uav_grid","schema_ok","coverage_tag",
               "status"]].round(2))
assert any(z[3]=="SCHEMA_OK" for z in candidates),"No valid probability candidate found."
print("IMPORTANT: current and backup files have identical raster *dimensions*;")
print("this says nothing about their actual valid-pixel coverage.")

## 02 — Sample valid-data coverage and candidate assessment

**CPU / read-only.** Sample the same image locations for available probability products. A missing backup must be reported as missing, not recreated or silently substituted. Sampling is a screening diagnostic, **not** a definitive full-mosaic coverage measurement.

**Observed run, 11 October 2026:** current maps exist for all five sites and have correct native-grid metadata; the four pre-ROI backup files were not found at the expected names, and the independent full-Luki mosaic is still pending. Proceed with the audit, then run NB04 Block 09 for full Luki inference. Validate whether the four other maps have the desired whole-mosaic coverage before generating satellite target fractions.

In [ ]:
# BLOCK 02 — Stratified same-location probability coverage comparison
# Fast read-only sample; NOT an exact entire-mosaic valid-area computation.
# Sample 12 x 12 source-coordinate locations with 128-px windows each.
GRID_N = 12
SAMPLE_PX = 128
sample_rows=[]
for site in SITES:
    if not RGB[site].exists():continue
    site_candidates=[(kind,path) for s,kind,path,status in candidates
                     if s==site and status=="SCHEMA_OK"]
    if not site_candidates:continue
    with rasterio.open(RGB[site]) as src:
        locations=[]
        for yf in (np.arange(GRID_N)+.5)/GRID_N:
            for xf in (np.arange(GRID_N)+.5)/GRID_N:
                row=int(yf*src.height);col=int(xf*src.width)
                row=max(0,min(row-SAMPLE_PX//2,src.height-SAMPLE_PX))
                col=max(0,min(col-SAMPLE_PX//2,src.width-SAMPLE_PX))
                locations.append(Window(col,row,min(SAMPLE_PX,src.width),min(SAMPLE_PX,src.height)))
        area_ha=src.width*src.height*abs(
            src.transform.a*src.transform.e-src.transform.b*src.transform.d)/1e4
    for kind,path in site_candidates:
        n=0;valid_total=0;invalid_encoding=0;mean_sum=0.0
        counts=[]
        with rasterio.open(path) as ds:
            for w in locations:
                a=ds.read(1,window=w)
                good=(a!=NODATA)&(a<=SCALE)
                invalid_encoding+=int(((a>NODATA)|( (a>SCALE)&(a!=NODATA))).sum())
                n+=a.size
                valid_total+=int(good.sum())
                if good.any():mean_sum+=float(a[good].astype(np.float64).sum())
                counts.append(float(good.mean()))
        frac=valid_total/max(n,1)
        sample_rows.append({
            "site":site,"candidate_kind":kind,"path":str(path),
            "sampled_pixels":n,"sample_valid_pixels":valid_total,
            "sample_valid_fraction":frac,
            "sample_mean_probability":mean_sum/valid_total/SCALE if valid_total else np.nan,
            "sample_invalid_encoding_pixels":invalid_encoding,
            "approx_valid_ha_NOT_EXACT":area_ha*frac,
            "sample_window_valid_fraction_min":min(counts),
            "sample_window_valid_fraction_max":max(counts),
        })
coverage=pd.DataFrame(sample_rows)
coverage.to_csv(VALIDATION/"02_stratified_coverage.csv",index=False)
display(coverage[["site","candidate_kind","sample_valid_fraction",
                  "sample_mean_probability","sample_window_valid_fraction_min",
                  "sample_window_valid_fraction_max","sample_invalid_encoding_pixels"]].round(4))
assert (coverage.sample_invalid_encoding_pixels==0).all(),"Out-of-range probabilities found."
print("Sample coverage is indicative only. ROI masks, orthomosaic edges and clustering")
print("mean we must NOT use the approximate area as exact valid-imagery area.")

# Select the broadest available candidate per site for **inspection only**.
# Final acceptance requires rigorous coverage analysis and source provenance.
rank={"full_mosaic":0,"pre_ROI_backup":1,"current":2}
coverage["priority"]=coverage.candidate_kind.map(rank)
selected=(coverage.sort_values(["site","sample_valid_fraction","priority"],
                              ascending=[True,False,True])
                  .drop_duplicates("site").copy())
selected["audit_decision"]="INSPECT_REQUIRED"
selected["reason"]="Coverage and full-mosaic provenance not yet independently certified"
selected.to_csv(VALIDATION/"02_candidate_selection_for_review.csv",index=False)
print("\nRECOMMENDED CANDIDATES FOR COVERAGE REVIEW (not accepted as FULL):")
display(selected[["site","candidate_kind","sample_valid_fraction","audit_decision"]].round(4))
missing=set(SITES)-set(selected.site)
if missing:print("MISSING candidate sites:",sorted(missing))
print("DATA GATE: full-mosaic status and satellite grid alignment remain unresolved.")
print("No satellite target grids or training tables have been generated.")

## 03 — UAV RGB footprint versus U-Net prediction coverage

**Critical audit step (CPU/read-only).** On the same sampled spatial locations, compare valid RGB coverage to valid probability coverage. This avoids concluding that sparse predictions are incomplete when much of the orthomosaic is background. The RGB proxy excludes exact black, white and blue pixels; its reliability must be visually checked, and Luki's defective fourth-band raster mask must not be used to discard actual valid RGB.

In [ ]:
# BLOCK 03 — Compare U-Net probability coverage against valid UAV RGB (CPU/read-only).
# Same 12x12 stratified locations as Block 02, larger 256-pixel windows.
# No requirement to load the full very large orthomosaics into memory.
from rasterio.windows import Window
import numpy as np,pandas as pd,rasterio
RGB_SAMPLE_PX=256
RGB_GRID_N=12
res=[]
for site in SITES:
    ip=RGB[site]
    if not ip.exists():continue
    opts=[("current",CURRENT[site])]
    if site in BACKUP:opts.append(("pre_ROI_backup",BACKUP[site]))
    if site=="Luki":opts.append(("full_mosaic",FULL_LUKI))
    opts=[(name,p) for name,p in opts if p.exists()]
    if not opts:continue
    with rasterio.open(ip) as src:
        assert src.count>=3,f"{site}: expected RGB bands"
        for variant,pp in opts:
            counts={x:0 for x in ("sample_pixels","rgb_nonbackground","rgb_dataset_mask",
                                  "prob_valid","both_nonbackground_and_prob","rgb_without_prob",
                                  "prob_without_rgb","nonbg_black","nonbg_white","nonbg_blue")}
            with rasterio.open(pp) as prob:
                assert prob.width==src.width and prob.height==src.height
                for yi in range(RGB_GRID_N):
                    for xi in range(RGB_GRID_N):
                        rr=max(0,min(int((yi+.5)/RGB_GRID_N*src.height)-RGB_SAMPLE_PX//2,
                                     src.height-RGB_SAMPLE_PX))
                        cc=max(0,min(int((xi+.5)/RGB_GRID_N*src.width)-RGB_SAMPLE_PX//2,
                                     src.width-RGB_SAMPLE_PX))
                        win=Window(cc,rr,min(src.width,RGB_SAMPLE_PX),min(src.height,RGB_SAMPLE_PX))
                        rgb=src.read([1,2,3],window=win)
                        pred=prob.read(1,window=win)
                        black=np.all(rgb==0,axis=0)
                        white=np.all(rgb==255,axis=0)
                        blue=(rgb[0]==0)&(rgb[1]==0)&(rgb[2]==255)
                        # Conservative data-based proxy. Black/white/blue can be real
                        # reflectance values, so compare with raster mask and visually QA.
                        image_ok=~(black|white|blue)
                        declared_ok=src.dataset_mask(window=win)>0
                        pred_ok=(pred!=65535)&(pred<=10000)
                        counts["sample_pixels"]+=int(pred.size)
                        counts["rgb_nonbackground"]+=int(image_ok.sum())
                        counts["rgb_dataset_mask"]+=int(declared_ok.sum())
                        counts["prob_valid"]+=int(pred_ok.sum())
                        counts["both_nonbackground_and_prob"]+=int((image_ok&pred_ok).sum())
                        counts["rgb_without_prob"]+=int((image_ok&~pred_ok).sum())
                        counts["prob_without_rgb"]+=int((~image_ok&pred_ok).sum())
                        counts["nonbg_black"]+=int(black.sum())
                        counts["nonbg_white"]+=int(white.sum())
                        counts["nonbg_blue"]+=int(blue.sum())
            c=counts;n=max(c["sample_pixels"],1)
            valid_rgb=max(c["rgb_nonbackground"],1)
            res.append(dict(site=site,variant=variant,**counts,
                rgb_nonbackground_fraction=c["rgb_nonbackground"]/n,
                rgb_dataset_mask_fraction=c["rgb_dataset_mask"]/n,
                probability_fraction_of_bbox=c["prob_valid"]/n,
                probability_coverage_of_valid_RGB=c["both_nonbackground_and_prob"]/valid_rgb,
                valid_RGB_missing_prediction_fraction=c["rgb_without_prob"]/valid_rgb,
                prediction_outside_RGB_proxy_fraction=c["prob_without_rgb"]/max(c["prob_valid"],1)))
q=pd.DataFrame(res)
q.to_csv(VALIDATION/"03_RGB_vs_probability_coverage.csv",index=False)
view=["site","variant","rgb_nonbackground_fraction","rgb_dataset_mask_fraction",
      "probability_fraction_of_bbox","probability_coverage_of_valid_RGB",
      "valid_RGB_missing_prediction_fraction","prediction_outside_RGB_proxy_fraction"]
display(q[view].round(4))
if len(q):
    for row in q.itertuples():
        if row.probability_coverage_of_valid_RGB<0.90:
            print(f"CHECK {row.site} [{row.variant}]: "
                  f"only {row.probability_coverage_of_valid_RGB:.1%} sampled valid RGB covered")
print("IMPORTANT: This is a STRATIFIED SAMPLE, not an exact area audit.")
print("RGB validity is a heuristic (exact black, white and blue excluded).")
print("Luki dataset_mask may be zero because its fourth band is undefined/nodata=255.")
print("NEXT GATE: inspect the results and a visual RGB + prediction overlay before deciding on further inference.")


## 04 — Five-site FULL production map acceptance QA

**Production complete (11 Oct 2026):** 5/5 full-resolution UAV mosaics have been inferred using the frozen checkpoint; the existing ROI-limited products remain unchanged. The full outputs are the authoritative reference inputs for satellite target generation. **Run Block 04 after Block 00** to independently check the new products' georeferencing, schema, production provenance and pixel-level probability coverage against the exact RGB-background validity convention used in NB04 Block 10. Sampling is deliberately bounded (12 × 12 × 256² pixels per site). A PASS does not replace complete spatial QA or visualization. Luki remains in EPSG:32733; other sites EPSG:32635.

In [ ]:
# BLOCK 04 — Verify authoritative FULL-mosaic products (CPU-only, read-only).
# Fast stratified paired RGB/probability pixel QA; no model execution.
import json
from pathlib import Path
import numpy as np,pandas as pd,rasterio
from rasterio.windows import Window
from IPython.display import display

FULL = {site: MAPS/f"{site}_musanga_probability_uint16_FULL.tif" for site in SITES}
REPORT = VALIDATION/"04_full_mosaic_production_QA.csv"
EXPECTED_MODEL = "final_unet_resnet18_2a5a0fb4d0de_ep04.pt"
rows=[]
GRID4=12
SIZE4=256
for site in SITES:
    rgb_path=RGB[site];prob_path=FULL[site]
    rec=dict(site=site,full_map=str(prob_path),full_exists=prob_path.exists())
    if not rgb_path.exists() or not prob_path.exists():
        rec["status"]="MISSING_RGB_OR_FULL_MAP";rows.append(rec);continue
    with rasterio.open(rgb_path) as src,rasterio.open(prob_path) as pred:
        same_grid=(src.width==pred.width and src.height==pred.height and
                   src.crs==pred.crs and src.transform==pred.transform)
        schema=(pred.count==1 and pred.dtypes[0]=="uint16" and pred.nodata==65535)
        tags=pred.tags()
        tag_ok=(tags.get("COVERAGE")=="FULL_VALID_RGB" and
                tags.get("COMPLETE")=="YES" and
                tags.get("MODEL_CHECKPOINT")==EXPECTED_MODEL and
                tags.get("MUSANGA_SCALE")=="10000" and
                bool(tags.get("SPEC_FINGERPRINT")))
        rec.update(crs=str(src.crs),same_grid=same_grid,schema_ok=schema,
                   provenance_ok=tag_ok,model=tags.get("MODEL_CHECKPOINT"),
                   fingerprint=tags.get("SPEC_FINGERPRINT"),
                   full_file_GB=round(prob_path.stat().st_size/1e9,3))
        if not (same_grid and schema and tag_ok):
            rec["status"]="METADATA_FAIL";rows.append(rec);continue
        counts=dict(n=0,rgb_valid=0,prob_valid=0,both=0,missing=0,
                    excess=0,illegal=0,special_dataset_valid=0)
        for iy in range(GRID4):
            for ix in range(GRID4):
                h=min(SIZE4,src.height);w=min(SIZE4,src.width)
                rr=max(0,min(int((iy+.5)*src.height/GRID4)-h//2,src.height-h))
                cc=max(0,min(int((ix+.5)*src.width/GRID4)-w//2,src.width-w))
                win=Window(cc,rr,w,h)
                rgb=src.read([1,2,3],window=win)
                black=np.all(rgb==0,axis=0)
                white=np.all(rgb==255,axis=0)
                blue=(rgb[0]==0)&(rgb[1]==0)&(rgb[2]==255)
                valid_rgb=~(black|white|blue)
                p=pred.read(1,window=win)
                valid_p=(p<=10000)
                counts["n"]+=p.size
                counts["rgb_valid"]+=int(valid_rgb.sum())
                counts["prob_valid"]+=int(valid_p.sum())
                counts["both"]+=int((valid_rgb&valid_p).sum())
                counts["missing"]+=int((valid_rgb&~valid_p).sum())
                counts["excess"]+=int((~valid_rgb&valid_p).sum())
                counts["illegal"]+=int(((p>10000)&(p!=65535)).sum())
                counts["special_dataset_valid"]+=int((src.dataset_mask(window=win)>0).sum())
        rec.update(rgb_valid_sample_fraction=counts["rgb_valid"]/counts["n"],
                   probability_valid_sample_fraction=counts["prob_valid"]/counts["n"],
                   probability_coverage_given_RGB=counts["both"]/max(counts["rgb_valid"],1),
                   probability_without_RGB_fraction=counts["excess"]/max(counts["n"],1),
                   missing_RGB_pixels=counts["missing"],
                   illegal_prediction_pixels=counts["illegal"],
                   dataset_mask_sample_fraction=counts["special_dataset_valid"]/counts["n"])
        rec["status"]=("PASS_SAMPLED" if counts["missing"]==0 and counts["excess"]==0
                       and counts["illegal"]==0 else "CHECK_SAMPLE_MISMATCH")
        rows.append(rec)
df_full=pd.DataFrame(rows)
df_full.to_csv(REPORT,index=False)
show=["site","crs","full_file_GB","rgb_valid_sample_fraction",
      "probability_valid_sample_fraction","probability_coverage_given_RGB",
      "missing_RGB_pixels","illegal_prediction_pixels","status"]
display(df_full[[x for x in show if x in df_full]].round(5))
print("FULL-MOSAIC PRODUCTS:",int(df_full.full_exists.sum()),"/",len(SITES))
print("Sample status counts:",df_full.status.value_counts().to_dict())
print("Source rasters not modified. This verifies sampled equality, not yet exact whole-raster equality.")
print("NEXT: determine Sentinel-2 acquisition dates, exact pixel grids, and embedding native transforms.")


## 05 — Scientific design and modelling contract

### UAV reference target
- Create a 5 m intermediate aggregation grid per site, then align satellite-scale targets to a verified Sentinel-2 projection and pixel origin from *full-coverage* UAV maps, with pixel area weighting where necessary.
- Maintain **valid UAV observation fraction** and **mean Musanga probability** as separate bands. Derive cover-fraction alternatives based on calibrated or fixed U-Net probability thresholds; never treat unmapped pixels as zero.
- Consider crown labels / Luki inventory for independent UAV-prediction calibration and audit. Thresholds must not be optimized using held-out satellite sites.
- Require a minimum observed high-resolution area per satellite grid cell (initial proposal ≥90%; sensitivity: 70%, 80%, 95%).
- Use strictly aligned support grids at 10, 30, 50, 75 and 100 m, all integer multiples of 5 m. Do not interpolate a 10 m satellite measurement into novel 10 m independent observations.

### Satellite feature families (independently ablated)
- **S00** geographic/climatological null baseline only; avoid site identifiers as predictors in site-held-out transfer.
- **S01** Sentinel-2 SR: B2/B3/B4/B5/B6/B7/B8/B8A/B11/B12 and NDVI, EVI, NDRE, NBR, NDMI; use correct resampling of native 20 m bands.
- **S02** Sentinel-2 quality-filtered seasonal percentiles and cloud-free acquisition counts, matched to UAV acquisition epoch.
- **S03** Sentinel-2 monthly/quarterly trajectories, phenological amplitudes and long-term trend; explicitly check missing periods and date leakage.
- **E01** Earth-observation embeddings (e.g. annual Sentinel-2-based embeddings), with provenance, source year and spatial support recorded.
- **F01** spectral + temporal; **F02** embeddings + spectral; **F03** all combined.
- Optional later: Sentinel-1, terrain, forest disturbance history. Keep separate so contribution is identifiable.

### Resolution experiment
Model 10, 30, 50, 75, 100 m *support* with the same site holdouts and comparable coverage gates. Distinguish (i) physical aggregation of UAV fraction, (ii) satellite native spatial resolution, (iii) smoothing/resampling. A 10 m multispectral feature stack contains native 20 m bands and must be labelled accordingly.

### Validation design
**Primary:** five leave-one-site-out folds, training/tuning on four sites; avoid using Luki for any model choice in its held-out fold.
**Secondary:** buffered spatial-block cross-validation within the training sites, blocks substantially larger than the 100 m response grid (test 500 m / 1 km based on autocorrelation).
**No leakage:** tune preprocessing, feature selection, threshold calibration and hyperparameters inside each training fold. Sample weights should avoid dominance by large mosaics and spatially autocorrelated pixels.

### Algorithms
Reference baselines: constant mean, ridge/elastic net and random forest. Main models: HistGradientBoosting or LightGBM plus a regularized embedding model. Optional spatial CNN only after tabular transfer baseline is established. Predict bounded fractions via suitable link/loss or calibration.

### Metrics and science
Primary: site-held-out MAE and RMSE (fraction units), calibration bias and R² relative to within-site variance. Secondary: Spearman ranking, occurrence detection AUC/AP where prevalence is relevant, abundance-stratified error, quality-/coverage-stratified error and bootstrap confidence intervals by spatial block. Report site-by-site and macro averages; do not use random pixel-split scores as external transfer estimates.

### Expected outputs
- Reproducible coverage-checked 5 m UAV reference grids, plus aggregate target grids at all five scales.
- Five-site harmonized spectral/temporal/embedding feature tables and provenance.
- Predeclared experiment matrix and leak-proof fold manifests.
- LO(S)O benchmark, feature ablations, support-scale curves, SHAP/permutation-importance checks on held-out data.
- Final full-predictor model and uncertainty rasters, then ecological tests against disturbance/succession.

## 06 — Prespecified experiment registry

**Scaffold only.** Register resolution × feature-family × algorithm combinations. Do not fit models before reference coverage and satellite grid alignment have passed QA.

In [ ]:
# BLOCK 06 — Prespecified experiment registry (scaffold; no model fitting)
resolutions=[10,30,50,75,100]
feature_families=["S01_S2_static","S02_S2_seasonal","S03_S2_trajectory",
                  "E01_embeddings","F01_spectral_temporal","F02_embedding_spectral","F03_all"]
algorithms=["ridge","random_forest","hist_gradient_boosting"]
experiments=pd.DataFrame(
    [(r,f,a) for r in resolutions for f in feature_families for a in algorithms],
    columns=["resolution_m","feature_family","algorithm"]
)
experiments["status"]="NOT_RUN"
experiments.to_csv(OUT/"experiment_registry.csv",index=False)
print("Registered experiments:",len(experiments))
print("× 5 independent site holdouts; not fitted until data QA passes.")
display(experiments.head(12))

## 07 — Inspect exact Sentinel-2 and embedding grids (Earth Engine, read-only)

**Run Block 00, then Block 07.** This connects to Earth Engine project `musanga-511110` and probes the **actual native projection and six-element affine pixel transform** of 2024 Sentinel-2 surface reflectance B2 (10 m) and Google Satellite Embedding V1 annual A00 (10 m), at the geographic centre of each UAV mosaic. The year **2024 is a provisional grid probe only**, not a scientific selection of the satellite observation period. The embedding product contains 64 axes. Document every site's native CRS and whether satellite origins agree; do **not** assume 10 m S2 and embeddings are co-registered merely because nominal resolutions match.

**Next gate:** verify UAV acquisition dates, check whether a site overlaps more than one S2/embedding tile, choose a site-specific canonical 10 m grid and aligned 30/50/75/100 m grids. **75 m cannot be an exact integer aggregation of 10 m cells**, so derive each scale from the high-resolution UAV support with consistent geographical anchors rather than claiming nested 10 m blocks. This block only reads metadata, writes a compact audit CSV/JSON, and does not create target rasters.

In [ ]:
# BLOCK 07 — Earth Engine satellite pixel-grid discovery (read-only, no exports).
# Run Block 00 first. Requires access to project musanga-511110.
# Calendar year 2024 is used ONLY to inspect grid geometry, not to select modelling dates.
import sys,subprocess,json,math
from pathlib import Path
import numpy as np,pandas as pd,rasterio
from rasterio.warp import transform_bounds
from IPython.display import display
try:
    import ee
except ImportError:
    subprocess.run([sys.executable,"-m","pip","-q","install","earthengine-api"],check=True)
    import ee
EE_PROJECT07="musanga-511110"
try:
    ee.Initialize(project=EE_PROJECT07)
except Exception as err:
    print("Earth Engine authorization required:",str(err)[:300])
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT07)
S2_COLLECTION07="COPERNICUS/S2_SR_HARMONIZED"
EMB_COLLECTION07="GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL"
GRID_YEAR07=2024
start07=f"{GRID_YEAR07}-01-01";end07=f"{GRID_YEAR07+1}-01-01"
info07=[]
for site in SITES:
    with rasterio.open(RGB[site]) as src:
        bounds_ll=transform_bounds(src.crs,"EPSG:4326",*src.bounds,densify_pts=21)
        cx=(bounds_ll[0]+bounds_ll[2])/2
        cy=(bounds_ll[1]+bounds_ll[3])/2
        uv=dict(site=site,uav_crs=str(src.crs),uav_res_x=abs(src.transform.a),
                uav_res_y=abs(src.transform.e),lon=cx,lat=cy,
                uav_bounds_lonlat=list(bounds_ll))
    point=ee.Geometry.Point([cx,cy])
    row=dict(uv,s2_collection=S2_COLLECTION07,
             embedding_collection=EMB_COLLECTION07,grid_year=GRID_YEAR07)
    for prefix,col,band in (
        ("s2",S2_COLLECTION07,"B2"),
        ("embedding",EMB_COLLECTION07,"A00")):
        try:
            fc=(ee.ImageCollection(col).filterDate(start07,end07)
                .filterBounds(point))
            count=int(fc.size().getInfo())
            row[f"{prefix}_images_point_year"]=count
            if count==0:
                row[f"{prefix}_status"]="NO_IMAGES"
                continue
            im=ee.Image(fc.first())
            proj=im.select(band).projection().getInfo()
            row[f"{prefix}_status"]="OK"
            row[f"{prefix}_crs"]=proj.get("crs")
            row[f"{prefix}_transform"]=proj.get("transform")
            row[f"{prefix}_image_id"]=im.get("system:index").getInfo()
        except Exception as error:
            row[f"{prefix}_status"]="ERROR"
            row[f"{prefix}_error"]=str(error)[:350]
    s=row.get("s2_transform");e=row.get("embedding_transform")
    if isinstance(s,list) and isinstance(e,list) and len(s)==6 and len(e)==6:
        same_crs=row.get("s2_crs")==row.get("embedding_crs")
        # Compare all 6 affine elements. Zero fractional pixel offsets is a
        # necessary but not sufficient test for the chosen scene's alignment.
        aligned=(same_crs and np.allclose(s,e,rtol=0,atol=1e-7))
        row["same_exact_grid"]=bool(aligned)
        if same_crs and s[0] and s[4]:
            row["origin_offset_s2_pixels_x"]=(e[2]-s[2])/s[0]
            row["origin_offset_s2_pixels_y"]=(e[5]-s[5])/s[4]
            row["integer_pixel_shift_only"]=bool(
                np.allclose([row["origin_offset_s2_pixels_x"],
                             row["origin_offset_s2_pixels_y"]],
                            np.rint([row["origin_offset_s2_pixels_x"],
                                     row["origin_offset_s2_pixels_y"]]),
                            atol=1e-6))
    info07.append(row)
    print(site,"| S2:",row.get("s2_status"),
          row.get("s2_crs",""),"| embeddings:",
          row.get("embedding_status"),row.get("embedding_crs",""),
          "| exact match:",row.get("same_exact_grid","UNKNOWN"))
df_grid07=pd.DataFrame(info07)
out07=VALIDATION/"07_satellite_grid_discovery_2024"
df_grid07.to_csv(str(out07)+".csv",index=False)
Path(str(out07)+".json").write_text(json.dumps(info07,indent=2))
cols07=["site","uav_crs","s2_crs","embedding_crs","s2_images_point_year",
        "embedding_images_point_year","same_exact_grid",
        "origin_offset_s2_pixels_x","origin_offset_s2_pixels_y",
        "integer_pixel_shift_only","s2_status","embedding_status"]
display(df_grid07[[a for a in cols07 if a in df_grid07.columns]])
print("Dataset dates here are grid-only probes, not UAV acquisition dates.")
print("NOTE: a point sample does NOT certify every S2/embedding tile across each mosaic.")
print("No target pixels or satellite training features have been exported.")


## 08 — UAV acquisition-year register and temporal matching gate

**Confirmed project years:** Yoko 2020; MOS 2024; Baego 2023; Yangambi 2024 and **Luki 2023** (both user-confirmed). Luki is also supported by Plumacker et al. (2026), *Toward Consistent Canopy Characterization From Low-Cost UAV Imagery*, DOI: 10.1002/ece3.74260. Supplementary Figure S3 documents orthomosaics dated **4 May 2023** and **1 November 2023**. The year of `Luki.tif` is confirmed as 2023; its exact acquisition day remains undetermined. Field measurements were 2–24 April 2023.

Additional 2024 mosaics YB_Carlier, YB_Forgeo, and YB_Joseph are noted separately; they are not silently treated as the `Yangambi_georef.tif` source.

**Temporal protocol:** use acquisition-year Sentinel-2 and annual embeddings when available; then test seasonal and multiannual context. 2024 remains just a grid-probe year in Block 07, not a blanket predictor year. Verify satellite dataset availability in 2020, 2023 and 2024.


In [ ]:
# BLOCK 08 — Five-site acquisition years (provenance, CPU-only).
import pandas as pd
import json
UAV_ACQUISITION_YEAR = {
    "MOS_1_2": 2024,
    "Yoko": 2020,
    "Baego": 2023,
    "Yangambi": 2024,
    "Luki": 2023,
}
UAV_YEAR_SOURCE = {
    "MOS_1_2": "User acquisition table (Mos)",
    "Yoko": "User acquisition table",
    "Baego": "User acquisition table",
    "Yangambi": "User confirmation, 2026-10-11",
    "Luki": "User confirmation, 2026-10-11; supported by Plumacker et al. 2026, DOI:10.1002/ece3.74260",
}
UAV_DATE_NOTE = {
    "Luki": "Confirmed year 2023; exact date within year not verified (paper documents May and November dates)"
}
assert set(UAV_ACQUISITION_YEAR)==set(SITES)
years08=pd.DataFrame([
    dict(analysis_site=site,acquisition_year=UAV_ACQUISITION_YEAR[site],
         year_confirmed=True,source=UAV_YEAR_SOURCE[site],
         date_note=UAV_DATE_NOTE.get(site,""))
    for site in SITES
])
years08.to_csv(VALIDATION/"08_acquisition_years.csv",index=False)
display(years08)
print("All five annual acquisition years documented. Exact Luki acquisition DATE remains unresolved.")


## 09 — Uncorrected Sentinel-2 acquisition-year audit and site grid registry

**Run Blocks 00 and 08, then Block 09.** This reads only GEE `COPERNICUS/S2_SR_HARMONIZED` metadata for Yoko 2020, Baego and Luki 2023, MOS and Yangambi 2024. It defines the **native 10 m Sentinel-2 B2 pixel origin** for each site in its own UAV UTM CRS. It explicitly finds a **zone-35** S2 granule for Yangambi rather than selecting the first overlapping zone-34 granule; it uses all overlapping granules when later building composites. Nothing is exported or processed with BRDF. The result is a five-site 10 m grid manifest with year, CRS, dimensions, native tile and exact affine transform.

We select the actual S2 B2 pixel origin, not the embedding origin. The embedding grid can later be resampled to the same site grid. Any site lacking a scene in its UAV UTM CRS stops with a clear error rather than silently choosing the wrong zone.

In [ ]:
# BLOCK 09 — Year-matched uncorrected Sentinel-2 intake and canonical 10m grids
# CPU / Earth Engine; read-only metadata; no exports.
# Dependencies: run Blocks 00 and 08. Block 07 (2024 embedding probe) is NOT required.
import json, math, subprocess, sys
from pathlib import Path
import numpy as np, pandas as pd, rasterio
from rasterio.warp import transform_bounds
from IPython.display import display
try:
    import ee
except ImportError:
    subprocess.check_call([sys.executable,"-m","pip","-q","install","earthengine-api"])
    import ee
PROJECT09="musanga-511110"
try:
    ee.Initialize(project=PROJECT09)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT09)
assert "UAV_ACQUISITION_YEAR" in globals(), "Run Block 08 before Block 09."
assert set(UAV_ACQUISITION_YEAR)==set(SITES)
COLLECTION09="COPERNICUS/S2_SR_HARMONIZED"
S2_COLLECTION09=ee.ImageCollection(COLLECTION09)
REGISTRY09={}
REPORT09=[]
for site in SITES:
    year=int(UAV_ACQUISITION_YEAR[site])
    start=f"{year}-01-01";end=f"{year+1}-01-01"
    with rasterio.open(RGB[site]) as uav:
        assert uav.crs is not None and uav.crs.is_projected
        target_crs=str(uav.crs)
        bounds=[float(v) for v in uav.bounds]
        ll=transform_bounds(uav.crs,"EPSG:4326",*uav.bounds,densify_pts=21)
    region=ee.Geometry.Rectangle(list(ll),proj="EPSG:4326",geodesic=False)
    col=S2_COLLECTION09.filterDate(start,end).filterBounds(region)
    num=int(col.size().getInfo())
    if num==0:raise RuntimeError(f"{site}: no Sentinel-2 SR images in {year}")
    tiles=sorted(set(x for x in col.aggregate_array("MGRS_TILE").getInfo() if x))
    # Pick a native Sentinel-2 scene in the *same UTM zone* as the UAV.
    # Yangambi straddles S2 zones 34/35: never inherit the first zone-34 scene.
    zone=target_crs.split(":")[-1][-2:]
    preferred=[t for t in tiles if t.startswith(zone)]
    matched=[]
    for t in preferred:
        im=ee.Image(col.filter(ee.Filter.eq("MGRS_TILE",t)).first())
        p=im.select("B2").projection().getInfo()
        if p.get("crs")==target_crs:matched.append((t,p))
    if not matched:
        raise RuntimeError(
            f"{site}: no native S2 B2 tile on UAV CRS {target_crs}. "
            f"Available MGRS tiles: {tiles}. Do not silently choose a different UTM CRS.")
    chosen_tile,proj=matched[0]
    trans=proj["transform"]
    if not(len(trans)==6 and abs(trans[0]-10)<1e-8 and
           abs(trans[4]+10)<1e-8 and abs(trans[1])<1e-8 and abs(trans[3])<1e-8):
        raise RuntimeError(f"{site}: unexpected native S2 B2 grid: {proj}")
    # A canonical 10m window, snapped to exact S2 pixel origin, covering UAV bounds.
    ox,oy=float(trans[2]),float(trans[5])
    x0=ox+math.floor((bounds[0]-ox)/10)*10
    y0=oy-math.floor((oy-bounds[3])/10)*10
    ncols=math.ceil((bounds[2]-x0)/10)
    nrows=math.ceil((y0-bounds[1])/10)
    affine=[10,0,x0,0,-10,y0]
    # Relative integer scene-pixel shift, never a resampled arbitrary origin.
    assert np.allclose((x0-ox)/10,round((x0-ox)/10),atol=1e-5)
    assert np.allclose((y0-oy)/10,round((y0-oy)/10),atol=1e-5)
    registry=dict(site=site,uav_year=year,source_collection=COLLECTION09,
                  uav_crs=target_crs,s2_crs=proj["crs"],
                  s2_native_tile=chosen_tile,all_s2_tiles=tiles,
                  native_scene_transform=trans,
                  target_transform=affine,width=ncols,height=nrows,
                  uav_bbox_m=bounds,uav_bbox_lonlat=list(ll),
                  s2_granules_year=num,
                  note="S2 annual data only, NO BRDF adjustment; exact S2 B2 native grid")
    REGISTRY09[site]=registry
    REPORT09.append({k:registry[k] for k in (
        "site","uav_year","uav_crs","s2_native_tile","width","height","s2_granules_year")}|{
            "all_tiles":",".join(tiles),
            "target_x0":x0,"target_y0":y0})
    print(site, year, target_crs, "S2 scenes:",num,
          "tiles:",",".join(tiles),"selected:",chosen_tile,
          "grid:",ncols,"x",nrows)
pd.DataFrame(REPORT09).to_csv(VALIDATION/"09_s2_year_and_grid_registry.csv",index=False)
(VALIDATION/"09_s2_canonical_10m_grids.json").write_text(
    json.dumps(REGISTRY09,indent=2))
display(pd.DataFrame(REPORT09))
print("Success: native year-specific Sentinel-2 grid registry saved.")
print("No exports or BRDF processing performed.")


## 10 — Uncorrected Sentinel-2 annual composites and optional exports

**Run Block 10 after 00, 08, 09.** Constructs cloud/shadow-masked annual *uncorrected* Sentinel-2 SR predictors at the appropriate UAV year. Removes SCL NoData (0), saturated (1), shadow (3), unclassified/low confidence (7), clouds (8–9), cirrus (10), snow (11); retains dark area (2), vegetation (4), bare soil (5), water (6). No QA60 dependence (different availability across 2020–2024). Reflectances scaled 0.0001, 10 native optical bands, NDVI/NDMI/NBR/NDRE, and clear-observation count. The 20 m bands are bilinearly evaluated at the 10 m analysis grid; their native information support remains 20 m.

**By default this cell only assembles the lazy GEE images and reports their metadata.** To export five multiband georeferenced GeoTIFFs to Google Drive set `START_EXPORTS10=True` and rerun. All exports use site-specific canonical S2 CRS and `crsTransform` from Block 09, and no BRDF correction. Satellite fraction training and UAV aggregation will follow only after visual assessment of S2 coverage and exact grid outputs. Note that annual composites may conceal intraannual disturbances or species phenology; temporal experiments come later.

In [ ]:
# BLOCK 10 — Uncorrected Sentinel-2 SR annual feature composites (SCL cloud mask).
# Runs after 00, 08, 09. By default PREVIEW/QA ONLY, no export tasks started.
# Change START_EXPORTS10=True to create five georeferenced GeoTIFF Drive exports.
import ee, json, math, pandas as pd
assert "REGISTRY09" in globals() and set(REGISTRY09)==set(SITES)
S2_BANDS10_10M=["B2","B3","B4","B8"]
S2_BANDS10_20M=["B5","B6","B7","B8A","B11","B12"]
S2_BANDS10=S2_BANDS10_10M+S2_BANDS10_20M
EXCLUDED_SCL10=[0,1,3,7,8,9,10,11] # 2 dark pixels retained; see methodological note.
START_EXPORTS10=False
EXPORT_SITES10=list(SITES)
EXPORT_DRIVE_FOLDER10="Musanga_NB08_S2_uncorrected"
MAX_PIXELS10=1e10
def mask_s2_10(im):
    scl=im.select("SCL")
    clear=ee.Image(1)
    for cls in EXCLUDED_SCL10:
        clear=clear.And(scl.neq(cls))
    # 20/60m edge masks address occasional unmasked granule borders.
    clear=clear.And(im.select("B8A").mask()).And(im.select("B9").mask())
    spec10m=im.select(S2_BANDS10_10M)
    spec20m=im.select(S2_BANDS10_20M).resample("bilinear")
    return ee.Image.cat([spec10m,spec20m]).updateMask(clear).copyProperties(
        im,["system:time_start","MGRS_TILE","CLOUDY_PIXEL_PERCENTAGE"])
def annual_s2_10(site):
    row=REGISTRY09[site]
    y=row["uav_year"]
    region=ee.Geometry.Rectangle(row["uav_bbox_lonlat"],
                                 proj="EPSG:4326",geodesic=False)
    # IMPORTANT: retain all intersecting UTM granules, even at Yangambi.
    # Export onto the one site-specific canonical CRS and pixel grid.
    col=(ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
         .filterBounds(region).filterDate(f"{y}-01-01",f"{y+1}-01-01"))
    masked=col.map(mask_s2_10)
    annual=masked.median().multiply(0.0001)
    # 20m bands were marked for bilinear reprojection per source image,
    # not on the composite (Earth Engine composite resample is unsupported).
    spec=annual.select(S2_BANDS10)
    ndvi=spec.normalizedDifference(["B8","B4"]).rename("NDVI")
    ndmi=spec.normalizedDifference(["B8","B11"]).rename("NDMI")
    nbr=spec.normalizedDifference(["B8","B12"]).rename("NBR")
    ndre=spec.normalizedDifference(["B8A","B5"]).rename("NDRE")
    count=masked.select("B2").count().unmask(0).toFloat().rename("N_CLEAR_S2")
    # Count stays zero in cells without a valid observation; reflectance
    # layers remain masked (not falsely assigned zero).
    composite=(ee.Image.cat([spec,ndvi,ndmi,nbr,ndre,count])
               .toFloat().clip(region))
    return composite
COMPOSITES10={s:annual_s2_10(s) for s in SITES}
print("Prepared uncorrected year-matched S2 composites:")
for site in SITES:
    row=REGISTRY09[site]
    print(site,row["uav_year"],row["uav_crs"],
          f"{row['width']}x{row['height']} 10m grid",
          "bands:",len(S2_BANDS10)+5)
print("Cloud mask: SCL rejected",EXCLUDED_SCL10,
      "| SR scale 0.0001 | NO BRDF correction")
print("START_EXPORTS10 =",START_EXPORTS10)
if START_EXPORTS10:
    tasks10=[]
    for site in EXPORT_SITES10:
        r=REGISTRY09[site]
        im=COMPOSITES10[site].unmask(-9999,sameFootprint=False).toFloat()
        region=ee.Geometry.Rectangle(r["uav_bbox_lonlat"],
                                     proj="EPSG:4326",geodesic=False)
        task=ee.batch.Export.image.toDrive(
            image=im,description=f"NB08_S2_{site}_{r['uav_year']}_RAW",
            folder=EXPORT_DRIVE_FOLDER10,
            fileNamePrefix=f"{site}_S2_{r['uav_year']}_RAW_10m",
            region=region,crs=r["uav_crs"],
            crsTransform=r["target_transform"],
            maxPixels=MAX_PIXELS10,fileFormat="GeoTIFF",
            formatOptions={"cloudOptimized":True,"noData":-9999})
        task.start()
        tasks10.append(dict(site=site,task_id=task.id,
                            year=r["uav_year"],status="STARTED"))
        print("Started S2 export:",site,task.id)
    pd.DataFrame(tasks10).to_csv(VALIDATION/"10_s2_export_tasks.csv",index=False)
else:
    print("No Earth Engine tasks were launched. Review masks and grid QA,")
    print("then rerun with START_EXPORTS10=True when ready.")


## 11 — Aggregate full UAV probability maps to native Sentinel-2 10 m pixels

**Run Block 00, then Block 11.** Block 11 reads the canonical grid JSON saved by Block 09; it does **not** need Earth Engine authorization, the earlier S2 processing cells, or a GPU. All five full UAV U-Net maps produced in NB04 Block 10 are used. Original UAV files are preserved.

The output for each site is one **four-band GeoTIFF**, exactly matching the S2 10 m CRS, 6-element affine transform, grid width and height saved by Block 09:

1. `musanga_probability_mean`: mean U-Net probability (0–1) across **valid observed UAV area**.
2. `musanga_cover_fraction_p05`: proportion of valid observed UAV area classified Musanga at frozen threshold ≥0.5.
3. `uav_valid_fraction`: fractional valid RGB/U-Net coverage **of the entire 10 m cell**, including image boundaries; never assume missing UAV imagery means absence of Musanga.
4. `musanga_probability_sd`: spatial within-cell standard deviation of the U-Net probability.

Uses area-weighted **average reprojection** of separate probability numerator, squared-probability numerator, positive-threshold numerator and validity denominator; avoids biased interpolation at NoData edges. A **≥90% UAV-validity threshold** gates bands 1, 2 and 4, while band 3 remains present for all pixels. Band 4 is *spatial variability*, **not predictive uncertainty**. Output NoData is -9999; output paths are in `KVO_work/KVO_08_satellite_fraction/uav_targets_10m/`. Processing uses bounded destination tiles and local scratch, so no 4-billion-pixel raster is loaded into RAM. Existing outputs are verified by their processing fingerprints and never silently overwritten.

**Important limitation:** the ≥0.5 cover fraction is a classifier-derived canopy-cover proxy; mean probability is not automatically calibrated fractional abundance. Both are retained for experiments. Because full UAV inference covers the four training sites, their reference quality should be evaluated separately from the independent external Luki predictions.

**Runtime:** 32 × 32 target-cell windows may require substantial Drive I/O when reading many high-resolution pixels; use `RUN_SITES11` to process one site at a time if needed. If Colab interrupts midway through one site, rerun that site; completed sites are skipped.

In [ ]:
# BLOCK 11 — Full-resolution UAV probability -> exact S2 10m analysis grid
# CPU ONLY. Run Block 00 first. Block 09 canonical grid JSON must already be on Drive.
# No Earth Engine, GPU, model training, or modifications to input rasters.
from pathlib import Path
import json,hashlib,math,shutil,time,os
import numpy as np,pandas as pd,rasterio
from affine import Affine
from rasterio.windows import Window,from_bounds,bounds as window_bounds
from rasterio.warp import reproject,Resampling,transform_bounds
from tqdm.auto import tqdm
from IPython.display import display

GRID_FILE11=VALIDATION/"09_s2_canonical_10m_grids.json"
assert GRID_FILE11.is_file(),(
    f"Missing {GRID_FILE11}. Run Block 09 once to save exact S2 grids.")
GRIDS11=json.loads(GRID_FILE11.read_text())
assert set(SITES)==set(GRIDS11),"Canonical grid register does not contain all five sites."
FULL11={s:MAPS/f"{s}_musanga_probability_uint16_FULL.tif" for s in SITES}
DESTDIR11=OUT/"uav_targets_10m"
DESTDIR11.mkdir(parents=True,exist_ok=True)
RUN_SITES11=list(SITES) # set e.g. ["Luki"] to run one site
TILE11=32             # 32x32 target cells => bounded UAV reads, safe RAM
COVERAGE_MIN11=0.90   # gate reference target metrics below 90% observed UAV coverage
THRESHOLD11=0.50      # frozen pixel-level Musanga classification threshold
NODATA11=-9999.0
BANDS11=[
  "musanga_probability_mean",
  "musanga_cover_fraction_p05",
  "uav_valid_fraction",
  "musanga_probability_sd",
]
PIPELINE11="UAV_S2_10M_weighted_v1_mean_fraction_coverage_spatial_sd"
assert 0<COVERAGE_MIN11<=1 and 0<=THRESHOLD11<=1
assert set(RUN_SITES11).issubset(SITES)

def project_info11(site):
    g=GRIDS11[site]
    trans=Affine(*g["target_transform"])
    if not(np.isclose(trans.a,10) and np.isclose(trans.e,-10) and
           np.isclose(trans.b,0) and np.isclose(trans.d,0)):
        raise ValueError((site,"unexpected 10m S2 transform",g["target_transform"]))
    return g,trans

def run_target11(site):
    g,dst_transform=project_info11(site)
    source=FULL11[site]
    assert source.is_file(),f"{site}: full UAV probability map missing: {source}"
    assert RGB[site].is_file(),f"{site}: original UAV mosaic missing"
    with rasterio.open(source) as src:
        assert src.dtypes[0]=="uint16" and src.nodata==65535,(site,"invalid source encoding")
        assert str(src.crs)==g["uav_crs"],(site,src.crs,g["uav_crs"])
        assert src.tags().get("COMPLETE")=="YES" and src.tags().get("COVERAGE")=="FULL_VALID_RGB",(
            site,"full UAV map lacks production-complete metadata")
        assert src.tags().get("MUSANGA_SCALE")=="10000"
        src_signature=src.tags().get("SPEC_FINGERPRINT","")
        assert src_signature,(site,"no frozen-model provenance fingerprint")
        width=int(g["width"]);height=int(g["height"])
        crs=g["uav_crs"]
    fingerprint_input=dict(
        site=site,source_signature=src_signature,
        target_transform=list(g["target_transform"]),target_crs=crs,
        target_width=width,target_height=height,
        pipeline=PIPELINE11,threshold=THRESHOLD11,
        min_coverage=COVERAGE_MIN11
    )
    fingerprint=hashlib.sha256(json.dumps(fingerprint_input,sort_keys=True).encode()).hexdigest()[:20]
    final=DESTDIR11/f"{site}_musanga_uav_target_S2grid_10m.tif"
    if final.exists():
        with rasterio.open(final) as old:
            ok=(old.count==4 and old.width==width and old.height==height and
                old.crs==rasterio.crs.CRS.from_string(crs) and
                old.transform==dst_transform and old.nodata==NODATA11 and
                old.tags().get("FINGERPRINT")==fingerprint and
                old.tags().get("COMPLETE")=="YES")
        if not ok:
            raise RuntimeError(f"Existing target incompatible, inspect before re-running: {final}")
        print("Verified existing aligned UAV target, SKIP:",site)
        return dict(site=site,status="EXISTS",file=str(final),width=width,height=height)
    print(f"\n{site} | {width} x {height} | {crs} | 10 m S2 pixel grid")
    tmp=Path(f"/content/{site}_musanga_target_S2grid_10m_partial.tif")
    tmp.unlink(missing_ok=True)
    profile=dict(driver="GTiff",height=height,width=width,count=4,dtype="float32",
                 crs=crs,transform=dst_transform,nodata=NODATA11,
                 compress="deflate",predictor=3,tiled=True,
                 blockxsize=256,blockysize=256)
    # Use a local staged output then publish only once all windows complete.
    start=time.time();accepted=0;partial=0;observed=0;total=0
    try:
        with rasterio.open(source) as src, rasterio.open(tmp,"w+",**profile) as dst:
            for k,name in enumerate(BANDS11,1):
                dst.set_band_description(k,name)
            # Fill every output pixel with explicit NoData. This file is small (<10MB/site).
            for rr in range(0,height,256):
                for cc in range(0,width,256):
                    hh=min(256,height-rr);ww=min(256,width-cc)
                    initial=np.full((4,hh,ww),NODATA11,np.float32)
                    initial[2]=0.0   # no observed UAV pixels = zero valid-coverage fraction
                    dst.write(initial,window=Window(cc,rr,ww,hh))
            nrows=(height+TILE11-1)//TILE11
            for r0 in tqdm(range(0,height,TILE11),total=nrows,desc=f"{site} UAV->S2 10m rows"):
                for c0 in range(0,width,TILE11):
                    hh=min(TILE11,height-r0);ww=min(TILE11,width-c0)
                    target_win=Window(c0,r0,ww,hh)
                    tb=window_bounds(target_win,dst_transform)
                    # Reprojection footprint transformation also handles different CRS.
                    sb=transform_bounds(crs,src.crs,*tb,densify_pts=9)
                    sw=from_bounds(*sb,transform=src.transform)
                    margin=3
                    # Keep the FULL destination cell footprint in source coordinates,
                    # including pixels beyond the UAV raster boundary.
                    # Boundless 65535 pixels map to validity=0, allowing correct
                    # fractional edge-cell coverage rather than edge normalization.
                    sr0=int(math.floor(sw.row_off))-margin
                    sc0=int(math.floor(sw.col_off))-margin
                    sr1=int(math.ceil(sw.row_off+sw.height))+margin
                    sc1=int(math.ceil(sw.col_off+sw.width))+margin
                    if sr1<=0 or sc1<=0 or sr0>=src.height or sc0>=src.width:
                        continue
                    src_win=Window(sc0,sr0,sc1-sc0,sr1-sr0)
                    pix=src.read(1,window=src_win,boundless=True,fill_value=65535)
                    if not np.any(pix!=65535):
                        continue
                    local_transform=src.window_transform(src_win)
                    valid=pix!=65535
                    # All source-to-grid resampling is pixel-area averaging.
                    # For invalid pixels use zero numerator and zero validity.
                    # Divide by averaged validity rather than assign missing as absence.
                    work=np.zeros(pix.shape,dtype=np.float32)
                    work[valid]=pix[valid].astype(np.float32)/10000.0
                    def aggregate_mean11(array):
                        target=np.zeros((hh,ww),dtype=np.float32)
                        reproject(source=array,destination=target,
                                  src_transform=local_transform,src_crs=src.crs,
                                  dst_transform=dst.window_transform(target_win),
                                  dst_crs=dst.crs,src_nodata=None,dst_nodata=0,
                                  resampling=Resampling.average,num_threads=2,
                                  init_dest_nodata=True)
                        return target
                    fraction=aggregate_mean11(valid.astype(np.float32))
                    numerator=aggregate_mean11(work)
                    # Second moment and fixed probability >=0.5 indicator.
                    np.square(work,out=work)
                    square_num=aggregate_mean11(work)
                    work[:]=0
                    work[valid]=(pix[valid]>=int(round(THRESHOLD11*10000))).astype(np.float32)
                    cover_num=aggregate_mean11(work)
                    observed+=int((fraction>0).sum())
                    eligible=fraction>=COVERAGE_MIN11-1e-5
                    accepted+=int(eligible.sum())
                    partial+=int(((fraction>0)&~eligible).sum())
                    out=np.full((4,hh,ww),NODATA11,dtype=np.float32)
                    # valid coverage stays 0..1 even for cells not eligible for targets
                    out[2]=np.clip(fraction,0,1)
                    safe=np.maximum(fraction,1e-9)
                    mu=np.clip(numerator/safe,0,1)
                    variance=np.maximum(square_num/safe-mu*mu,0)
                    out[0,eligible]=mu[eligible]
                    out[1,eligible]=np.clip(cover_num/safe,0,1)[eligible]
                    out[3,eligible]=np.sqrt(variance[eligible])
                    dst.write(out,window=target_win)
                    total+=1
            dst.update_tags(
                COMPLETE="YES",FINGERPRINT=fingerprint,SOURCE_PROBABILITY=str(source),
                SOURCE_SPEC_FINGERPRINT=src_signature,ANALYSIS_GRID="native Sentinel-2 B2 10m",
                UAV_VALID_MIN=str(COVERAGE_MIN11),MUSANGA_THRESHOLD=str(THRESHOLD11),
                RESAMPLING="area average: numerator / valid area, non-observed pixels excluded",
                PROB_SD="spatial within-S2-cell variation, not model prediction uncertainty")
        with rasterio.open(tmp) as chk:
            assert chk.crs==rasterio.crs.CRS.from_string(crs)
            assert chk.transform==dst_transform
            assert chk.width==width and chk.height==height and chk.count==4
            assert chk.tags().get("FINGERPRINT")==fingerprint
            assert chk.descriptions==tuple(BANDS11)
        assert not final.exists()
        shutil.copy2(tmp,final)
        assert final.stat().st_size==tmp.stat().st_size
        tmp.unlink(missing_ok=True)
    except Exception:
        print("Failed site:",site,"Temporary partial output may exist:",tmp)
        raise
    qa=dict(site=site,status="COMPLETE",file=str(final),
            crs=crs,width=width,height=height,valid_reference_cells=accepted,
            partially_observed_cells_below_coverage_threshold=partial,
            sampled_or_partially_observed_cells=observed,
            min_coverage=COVERAGE_MIN11,threshold=THRESHOLD11,
            fingerprint=fingerprint,source_model_fingerprint=src_signature,
            runtime_minutes=round((time.time()-start)/60,2))
    (VALIDATION/f"11_{site}_target_10m_QA.json").write_text(json.dumps(qa,indent=2))
    print("COMPLETED",site,"accepted 10m cells:",accepted,
          "output MB:",round(final.stat().st_size/1e6,1))
    return qa

results11=[run_target11(s) for s in RUN_SITES11]
summary11=pd.DataFrame(results11)
summary11.to_csv(VALIDATION/"11_uav_target_10m_summary.csv",index=False)
display(summary11)
print("All requested 10m UAV targets complete; native UAV rasters unchanged.")


## 12 — Tropical-forest BRDF correction: published-code preflight (no exports)**Independent branch; keep RAW unchanged.** The provided Earth Engine JavaScript source (July 2026, PP; [original editor](https://code.earthengine.google.com/bf3975028f60f6c5a699a44691f127e6)) implements per-scene Ross–Thick + Li–Thin kernels and AMAP tropical-forest BRDF coefficients, with a Roy six-band alternative. The intended next implementation is a **BRDF-corrected copy of Block 10**, matching the same Sentinel-2 scenes, SCL masks, 10 m grid and output bands. Compare RAW vs AMAP and optionally Roy on exactly the same pixels/observations.**Important technical safeguards from source review:**- The author's example uses `COPERNICUS/S2_SR` and manually removes the Processing Baseline 04.00 radiometric offset. **Do not reapply that subtract-1000 operation to GEE's `COPERNICUS/S2_SR_HARMONIZED`**, which has already harmonized the reflectance scale.- The original exact viewing-angle algorithm depends on `users/wiell/SepalResources/sentinel2TileOrbits` and a match for every `MGRS_TILE` + `SENSING_ORBIT_NUMBER`. Verify that the asset is accessible and has the necessary records. The placeholder geometry used when a match is missing is *not* valid science; missing records must be excluded or block the corrected output.- AMAP provides coefficients for **B2,B3,B4,B5,B6,B7,B8,B11,B12** (nine bands), but **not B8A**, even though Block 10 exports B8A. Preserve B8A as uncorrected and flag it as such, or exclude B8A-dependent indices in the controlled BRDF comparison. Never claim that all ten S2 bands are AMAP-corrected.- BRDF adjustment is applied to each source image **before** making annual median composites. Scene-by-scene normalization must preserve cloud-masked support and band names, and report directional correction factors and missing-orbit rate.**Block 12 checks dependencies and saves the published AMAP/Roy coefficients and orbit lookup audit. It does not run BRDF correction, change RAW outputs, train any models or start Earth Engine tasks.** Run Blocks 00, 08, 09, then 12. If 09 variables are already present, only Block 12 is needed.


In [ ]:
# BLOCK 12 — Published AMAP BRDF implementation PRE-FLIGHT (read-only).# No exports; checks orbit lookup completeness and records exact coefficient provenance.# Requires Block 09 (REGISTRY09) in runtime.import json, pandas as pdfrom pathlib import Pathimport eeassert "REGISTRY09" in globals() and set(REGISTRY09)==set(SITES)try:    ee.Initialize(project="musanga-511110")except Exception:    ee.Authenticate()    ee.Initialize(project="musanga-511110")BRDF_ORBIT_ASSET12="users/wiell/SepalResources/sentinel2TileOrbits"# Coefficients taken verbatim from the user-supplied July 2026 PP GEE example.# Roy: 6-band reference; AMAP: 9-band tropical forest coefficients.COEFF12={ "Roy":{  "B2":[.0774,.0079,.0372],"B3":[.1306,.0178,.0580],  "B4":[.1690,.0227,.0574],"B8":[.3093,.0330,.1535],  "B11":[.3430,.0453,.1154],"B12":[.2658,.0387,.0639]}, "Amap":{  "B2":[.0497,.0323,-.0021],"B3":[.0603,.0204,.0734],  "B4":[.0258,-.0149,.1893],"B5":[.0856,.0173,.1397],  "B6":[.2985,.1560,-.2528],"B7":[.3781,.1926,-.3409],  "B8":[.3623,.1606,-.1811],"B11":[.2008,.0743,.1536],  "B12":[.0901,.0334,.0957]}}COEFF_FIELDS12=["fiso","fgeo","fvol"]B8A_POLICY12="UNCORRECTED: no AMAP coefficient in supplied author implementation"BRDF_CORRECT_COLLECTION12="COPERNICUS/S2_SR_HARMONIZED"BRDF_DO_NOT_SUBTRACT_OFFSET12=Trueprint("BRDF collection:",BRDF_CORRECT_COLLECTION12)print("Important: HARMONIZED offset already normalized; no -1000 DN adjustment.")print("AMAP corrected:",list(COEFF12["Amap"]))print("AMAP does NOT correct B8A:",B8A_POLICY12)asset_ok12=Falseorbit_fc12=Nonetry:    orbit_fc12=ee.FeatureCollection(BRDF_ORBIT_ASSET12)    sample12=orbit_fc12.limit(1).getInfo()    asset_ok12=bool(sample12.get("features"))    print("Orbit lookup accessible:",asset_ok12)    if asset_ok12:        print("Orbit table example properties:",list(sample12["features"][0].get("properties",{})))except Exception as ex:    print("ORBIT ASSET UNAVAILABLE:",str(ex)[:650])records12=[]for site in SITES:    cfg=REGISTRY09[site]    y=int(cfg["uav_year"])    geom=ee.Geometry.Rectangle(cfg["uav_bbox_lonlat"],                               proj="EPSG:4326",geodesic=False)    col=(ee.ImageCollection(BRDF_CORRECT_COLLECTION12)         .filterDate(f"{y}-01-01",f"{y+1}-01-01").filterBounds(geom))    # Audit tile/orbit combinations for every year-specific image, not just one.    try:        tile_orbits=col.select("B2").aggregate_array("MGRS_TILE").getInfo()        orbit_numbers=col.aggregate_array("SENSING_ORBIT_NUMBER").getInfo()        pairs=sorted(set((str(t),str(o)) for t,o in zip(tile_orbits,orbit_numbers)                         if t is not None and o is not None))        missing=[]        if asset_ok12:            for tile,orbit in pairs:                # Numeric orbit property in the author's source. Never compare numeric                # orbit number as a string in the server-side filter.                count=orbit_fc12.filter(ee.Filter.And(                    ee.Filter.eq("tileId",tile),                    ee.Filter.eq("orbitNumber",int(orbit)))).size().getInfo()                if int(count)==0:missing.append((tile,int(orbit)))        rec=dict(site=site,year=y,images=len(tile_orbits),                 distinct_tile_orbit_pairs=len(pairs),                 tile_orbit_pairs=json.dumps(pairs),                 orbit_asset_accessible=asset_ok12,                 missing_orbit_pairs=json.dumps(missing),                 missing_orbit_count=len(missing) if asset_ok12 else None,                 preflight=("PASS_ORBIT_LOOKUP" if asset_ok12 and not missing                            else "BLOCKED_MISSING_ORBITS" if asset_ok12                            else "BLOCKED_ORBIT_ASSET"))    except Exception as ex:        rec=dict(site=site,year=y,preflight="ERROR",error=str(ex)[:450])    records12.append(rec)qa12=pd.DataFrame(records12)display(qa12[["site","year","images","distinct_tile_orbit_pairs",              "orbit_asset_accessible","missing_orbit_count","preflight"]]        if all(x in qa12.columns for x in ["images","preflight"]) else qa12)qa12.to_csv(VALIDATION/"12_brdf_orbit_preflight.csv",index=False)(VALIDATION/"12_amap_roy_coefficients.json").write_text(    json.dumps({"fields":COEFF_FIELDS12,"coefficients":COEFF12,                "orbit_asset":BRDF_ORBIT_ASSET12,                "input_collection":BRDF_CORRECT_COLLECTION12,                "no_additional_baseline_offset":True,                "B8A_policy":B8A_POLICY12,                "source":"User-provided PP July 2026 Earth Engine JavaScript"},               indent=2))print("BRDF preflight complete — NO corrections or exports were run.")print("If orbit lookup is missing or inaccessible, do not publish a BRDF map")print("using placeholder viewing angles. Resolve this gate first.")
